# Google Gemini

Google's flagship family of multimodal LLMs (the **Gemini 2.5** generation: Pro, Flash, Flash-Lite), reached through the **Gemini API** via the unified `google-genai` Python SDK.

**Domain:** Proprietary Models & Coding AI  ·  **from study list**  ·  **runnable:** yes  ·  _needs API key (cells gate on `os.getenv`)_

## 1. What & Why

**Gemini** is Google's proprietary, closed-weight family of large language models. You don't download weights — you call a hosted API and pay per token. The current generation is **Gemini 2.5**, in three sizes:

| Model | Sweet spot |
|---|---|
| **Gemini 2.5 Pro** | Hardest reasoning, long-context analysis, agentic/coding work. Most expensive. |
| **Gemini 2.5 Flash** | The default workhorse — fast, cheap, strong. Use this unless you have a reason not to. |
| **Gemini 2.5 Flash-Lite** | Highest throughput / lowest cost for simple, high-volume classification & extraction. |

**The problem it solves:** you want frontier-quality text + vision + audio reasoning without training or hosting a model. Reach for Gemini specifically when you need:

- **Natively multimodal input** — text, images, audio, video, and PDFs go into the *same* request. Gemini was designed multimodal from the ground up rather than bolting vision on later.
- **Very long context** — up to ~1M input tokens on 2.5 Pro/Flash, so you can stuff whole codebases, long PDFs, or hours of transcript into a single call.
- **Built-in tools** — Google Search grounding, code execution, function calling, and structured (JSON-schema) output are first-class features.
- **A free tier + tight Google Cloud integration** — prototype free in Google AI Studio, then graduate the *same* code to Vertex AI for enterprise governance.

**When NOT to reach for it:** if you need on-prem / open weights, data that can never leave your VPC, or full fine-tuning control, use an open model (Llama, Qwen, Mistral). If you're already deep in the Anthropic or OpenAI ecosystem and don't need Gemini's multimodal or 1M-context edge, switching has little upside.

## 2. Mental Model

Think of Gemini as a **stateless multimodal function** behind an HTTP endpoint:

```
contents (text + images + audio + video + PDFs, in order)
        +  config (system instruction, temperature, tools, response_schema, thinking budget)
        |
        v
   [ Gemini 2.5 model ]
        |
        v
response.text   (and/or function calls, structured JSON, grounded citations)
```

Three things to internalize:

1. **Stateless.** The API remembers nothing between calls. "Chat history" is just you replaying the prior turns in `contents` every time. Multi-turn = a growing list of `{role, parts}` messages you send each request.
2. **Everything is `parts`.** A message is a list of parts, and a part can be text *or* inline/uploaded media. Multimodality isn't a special mode — it's just mixing part types in one list.
3. **`contents` in, `config` around it.** *What* you're asking about goes in `contents`; *how* the model should behave (system prompt, sampling, tools, output schema, reasoning effort) goes in `config`. Keeping these straight is most of the SDK.

## 3. Key Concepts

- **`google-genai` SDK** — the **current** unified Python library (`from google import genai`). It talks to both the Gemini Developer API and Vertex AI. The older `google-generativeai` package is **deprecated** — don't start new code on it.
- **`client.models.generate_content(model, contents, config)`** — the one call you use 90% of the time. Add `_stream` for token streaming, or use `client.chats` for managed multi-turn.
- **`contents`** — a string, a list of parts, or a list of `Content` messages (`role` = `"user"` / `"model"`). This is the conversation/input.
- **`GenerateContentConfig`** — knobs: `system_instruction`, `temperature`, `max_output_tokens`, `tools`, `response_mime_type` + `response_schema`, and `thinking_config`.
- **Multimodal parts** — pass `PIL.Image`, bytes via `types.Part.from_bytes(...)`, or large/ reused files via the **Files API** (`client.files.upload(...)`).
- **Structured output** — set `response_mime_type="application/json"` and a `response_schema` (a Pydantic model or dict) to force valid JSON back. No regex-scraping the prose.
- **Function calling / tools** — declare functions and the model emits a `function_call` part you execute and feed back. `Google Search` grounding and `code_execution` are built-in tools you just switch on.
- **Thinking** — 2.5 models reason before answering. `thinking_config` (a token "budget") trades latency/cost for harder reasoning; set budget to 0 to disable on Flash.
- **Context window** — ~1M input tokens (2.5 Pro/Flash). **Context caching** lets you reuse a large fixed prefix (a big PDF, a codebase) across calls at a discount.
- **Two front doors** — **Gemini API** (key from Google AI Studio, fastest to start) vs **Vertex AI** (same models, IAM/VPC/governance). Same SDK; flip a flag.

## 4. Setup

Install the **current** SDK (note: `google-genai`, *not* the deprecated `google-generativeai`):

```bash
pip install google-genai
```

Get a key from **[Google AI Studio](https://aistudio.google.com/apikey)** (free tier available) and export it. The SDK auto-reads `GEMINI_API_KEY` (or `GOOGLE_API_KEY`):

```bash
export GEMINI_API_KEY="your-key-here"
```

Then a minimal call is two lines:

```python
from google import genai
client = genai.Client()  # picks up the key from the environment
resp = client.models.generate_content(model="gemini-2.5-flash", contents="Hello")
print(resp.text)
```

The cells below run top-to-bottom in a fresh kernel **without** the package or a key installed — the live call is gated behind an `os.getenv` check.

In [ ]:
# This notebook is written to execute with or without the SDK / an API key.
# To run the live example, uncomment the install and set GEMINI_API_KEY first:
# %pip install google-genai

import os

have_key = bool(os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"))
print("GEMINI_API_KEY available:", have_key)
print("Default model we'll target:", "gemini-2.5-flash")

## 5. Worked Examples

### Example 1 — Estimate tokens & cost *before* you call (no network)

The two questions every Gemini integration eventually asks: "how many tokens is this?" and "what will it cost?" A rough `chars / 4` heuristic is plenty for back-of-envelope budgeting (use the real `client.models.count_tokens` when it matters).

In [ ]:
# Rough token + cost estimate for a Gemini 2.5 Flash call. Pure Python, no API.
# Prices are ILLUSTRATIVE (USD per 1M tokens) — always confirm at ai.google.dev/pricing.
PRICING = {
    "gemini-2.5-flash":      {"in": 0.30, "out": 2.50},
    "gemini-2.5-pro":        {"in": 1.25, "out": 10.00},
    "gemini-2.5-flash-lite": {"in": 0.10, "out": 0.40},
}

def est_tokens(text: str) -> int:
    # ~4 characters per token is a workable heuristic for English prose.
    return max(1, len(text) // 4)

def est_cost(model: str, in_tokens: int, out_tokens: int) -> float:
    p = PRICING[model]
    return in_tokens / 1e6 * p["in"] + out_tokens / 1e6 * p["out"]

prompt = "Summarize the theory of relativity for a 12-year-old in two sentences."
in_tok, out_tok = est_tokens(prompt), 60  # assume ~60-token reply

print(f"prompt: {prompt!r}")
print(f"input  ~{in_tok} tokens, output ~{out_tok} tokens\n")
for model in PRICING:
    print(f"  {model:<22} est ${est_cost(model, in_tok, out_tok):.6f}")

### Example 2 — Build a multimodal request, the way the SDK sees it (no network)

A single `contents` list can interleave text and media parts. Here we *build and inspect* the request structure (a system instruction + a text part + an image part) without sending it — so you can see exactly what a multimodal call is before paying for one.

In [ ]:
# Construct the request shape the SDK would send. Pure Python — no key, no network.
fake_image_bytes = b"\x89PNG\r\n\x1a\n...(pretend this is a real PNG)..."

request = {
    "model": "gemini-2.5-flash",
    "config": {
        "system_instruction": "You are a terse assistant. Answer in one sentence.",
        "temperature": 0.2,
        "max_output_tokens": 256,
    },
    "contents": [
        {"role": "user", "parts": [
            {"text": "What's in this image?"},
            {"inline_data": {"mime_type": "image/png", "bytes": len(fake_image_bytes)}},
        ]},
    ],
}

import json
print(json.dumps(request, indent=2))
print("\nText parts :", sum(1 for p in request["contents"][0]["parts"] if "text" in p))
print("Media parts:", sum(1 for p in request["contents"][0]["parts"] if "inline_data" in p))

### Example 3 — The real call, gated behind an API key

The actual SDK invocation. With a key set it returns live text; without one it prints the call shape so the notebook still executes cleanly end-to-end.

In [ ]:
import os

def call_gemini(prompt: str, model: str = "gemini-2.5-flash") -> str:
    from google import genai            # pip install google-genai
    from google.genai import types
    client = genai.Client()             # reads GEMINI_API_KEY / GOOGLE_API_KEY from env
    resp = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction="Reply with a single short word.",
            temperature=0.0,
            max_output_tokens=16,
        ),
    )
    return resp.text

if os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"):
    try:
        print("Gemini says:", call_gemini("Reply with exactly: pong").strip())
    except Exception as e:        # network/quota/SDK issues shouldn't break the notebook
        print("Live call failed:", type(e).__name__, e)
else:
    print("No API key set — skipping the live call.")
    print("Call shape: client.models.generate_content(")
    print("                model='gemini-2.5-flash', contents=..., config=...)")

### Example 4 — Force structured JSON output (call shape)

For extraction/classification you rarely want prose. Set `response_mime_type="application/json"` plus a `response_schema` (a Pydantic model works) and Gemini returns schema-valid JSON you can parse directly — no brittle string scraping.

In [ ]:
import os

def extract_person(text: str):
    from google import genai
    from google.genai import types
    from pydantic import BaseModel

    class Person(BaseModel):
        name: str
        age: int
        city: str

    client = genai.Client()
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Extract the person from: {text}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Person,
        ),
    )
    return resp.parsed  # already validated into a Person instance

sample = "Ada Lovelace, 36, lives in London."
if os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"):
    try:
        print(extract_person(sample))
    except Exception as e:
        print("Live call failed:", type(e).__name__, e)
else:
    print("No API key — showing the pattern instead.")
    print("response_mime_type='application/json' + response_schema=Person")
    print("=> resp.parsed is a validated Person(name=..., age=..., city=...)")

## 6. Gotchas & Pitfalls

- **Wrong SDK.** `google-generativeai` (with `genai.GenerativeModel(...)`) is the **deprecated** library. The current one is `google-genai` (`from google import genai`; `client.models.generate_content(...)`). Tutorials older than ~2024 use the dead one — its patterns won't map.
- **Thinking tokens cost money and add latency.** 2.5 models "think" before answering, and those reasoning tokens are billed as output. For simple, high-volume tasks set the thinking budget to 0 (Flash/Flash-Lite) or you'll pay for reasoning you didn't need.
- **`resp.text` can be `None`.** If the response was blocked by safety filters, hit `max_output_tokens`, or returned only a function call, `.text` is empty/`None`. Always check `resp.candidates[0].finish_reason` (e.g. `SAFETY`, `MAX_TOKENS`) before assuming you got prose.
- **Statelessness bites multi-turn.** The API holds no history. If you forget to resend prior turns in `contents` (or use `client.chats`), every message looks like turn one. Conversely, replaying a huge history re-bills all those input tokens each turn — use **context caching** for big fixed prefixes.
- **Free-tier data is used for training.** On the free Gemini API tier, prompts may be used to improve the product. Paid tier and Vertex AI are not. Don't paste anything sensitive into a free-tier key.
- **Rate limits + safety blocks aren't bugs.** Expect `429`/`RESOURCE_EXHAUSTED` and occasional safety blocks; wrap calls in retry-with-backoff and handle blocked responses gracefully rather than crashing.
- **Inline vs Files API.** Small media can go inline (base64) in the request, but it counts against request size. For large or reused files (big PDFs, video), upload via the **Files API** and reference the handle instead.
- **Model aliases drift.** Names like `gemini-2.5-flash` track to the latest stable snapshot, which can change behavior under you. Pin a dated snapshot when you need reproducibility.

## 7. When to Use vs Alternatives

| You need… | Reach for | Why |
|---|---|---|
| **Native multimodal** (image+audio+video+PDF in one call) | **Gemini 2.5 Pro/Flash** | Built multimodal from the start; widest input-type coverage. |
| **Huge context** (whole codebase / long video / many PDFs) | **Gemini 2.5 Pro** | ~1M-token window plus context caching for cheap reuse. |
| **Cheap, fast, high-volume** text tasks | **Gemini 2.5 Flash / Flash-Lite** | Excellent price/performance; Lite for the simplest jobs. |
| **Enterprise governance** (IAM, VPC-SC, data residency) | **Vertex AI** (same models) | Same SDK + Google Cloud controls. See the `google-vertex` notebook. |

**Honest trade-offs vs the competition:**

- **vs Anthropic Claude** — Claude is often preferred for careful coding, agentic tool-use, and instruction-following nuance; Gemini wins on multimodal breadth, the biggest context window, and free-tier accessibility. Many teams route by task.
- **vs OpenAI GPT** — comparable frontier quality and ecosystem maturity. GPT has a deeper third-party tooling/community; Gemini's edges are 1M context, native video/audio, and tight Google Cloud integration.
- **vs open models (Llama / Qwen / Mistral)** — choose open weights when you need on-prem/air-gapped deployment, full fine-tuning, no per-token cost at scale, or data that legally can't leave your infrastructure. You trade frontier multimodal quality and zero-ops convenience for control.

**Rule of thumb:** prototype on **Gemini 2.5 Flash** via the free Gemini API; escalate hard cases to **2.5 Pro**; move to **Vertex AI** when compliance demands it; switch to **open weights** only when hosting/control requirements force your hand.

## 8. Resources

- **Gemini API docs (official)** — https://ai.google.dev/gemini-api/docs
- **`google-genai` Python SDK reference** — https://googleapis.github.io/python-genai/
- **Google AI Studio (get a free API key, try prompts)** — https://aistudio.google.com/
- **Models & rate limits overview** — https://ai.google.dev/gemini-api/docs/models
- **Pricing** — https://ai.google.dev/gemini-api/docs/pricing
- **Structured output & function calling guide** — https://ai.google.dev/gemini-api/docs/structured-output
- **Vertex AI (enterprise front door, same models)** — https://cloud.google.com/vertex-ai/generative-ai/docs

**Related notebooks:** `google-vertex` (enterprise deployment of these same models), `anthropic-claude-api` and other entries in this domain for head-to-head comparison.